# 🛢️ SegFormer Oil Spill Segmentation trên SAR

**Notebook Kaggle — Chạy trực tiếp không cần cấu hình thêm**

| Thông tin | Chi tiết |
|-----------|----------|
| Model | SegFormer (MiT-B0 → B5 configurable) |
| Dataset | `deep-sar-oil-spill-segmentation-refined` |
| Task | Binary Segmentation: background vs oil spill |
| Framework | PyTorch + HuggingFace Transformers |
| GPU | Kaggle T4/P100 (16GB) |

---
**Cấu trúc Notebook:**
1. Setup & Install dependencies
2. Cấu hình đường dẫn Kaggle
3. EDA — Khám phá dataset
4. Tiền xử lý & Augmentation
5. Xây dựng model SegFormer
6. Training loop
7. Đánh giá & Visualization
8. Inference & Export

## 1. Setup & Install

In [ ]:
# Kaggle đã có: torch, torchvision, numpy, opencv, matplotlib, tqdm
# Chỉ cần cài thêm:
!pip install -q transformers>=4.35.0 albumentations>=1.3.1 huggingface_hub>=0.19.0
print('✓ Dependencies installed')

In [ ]:
import os
import sys
import csv
import glob
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation, SegformerConfig

# Seed để reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Cấu hình đường dẫn Kaggle

> Dataset Kaggle: [deep-sar-oil-spill-segmentation-refined](https://www.kaggle.com/datasets/)
> Thêm dataset này qua **+ Add Data** trước khi chạy notebook.

In [ ]:
# ── Kaggle dataset path ───────────────────────────────────────────────────────
# Kaggle mount dataset tại /kaggle/input/<dataset-slug>/
KAGGLE_INPUT = Path('/kaggle/input')

# Tìm tự động thư mục dataset
def find_dataset_root(base: Path, keywords: List[str] = ['deep-sar', 'oil-spill', 'sar']) -> Path:
    """Tự động tìm thư mục dataset trong /kaggle/input/"""
    for d in sorted(base.iterdir()):
        if any(k in d.name.lower() for k in keywords):
            return d
    # Fallback: lấy thư mục đầu tiên
    dirs = [d for d in base.iterdir() if d.is_dir()]
    if dirs:
        return dirs[0]
    raise FileNotFoundError(f'Không tìm thấy dataset trong {base}')

DATASET_ROOT = find_dataset_root(KAGGLE_INPUT)
print(f'Dataset root: {DATASET_ROOT}')
print('Contents:', [d.name for d in DATASET_ROOT.iterdir()])

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE        = (512, 512)        # (H, W) — resize target
BACKBONE        = 'nvidia/mit-b0'   # thay bằng mit-b2 cho accuracy tốt hơn
NUM_CLASSES     = 2                 # 0: background, 1: oil spill
PRETRAINED      = True

BATCH_SIZE      = 4                 # giảm xuống 2 nếu OOM với backbone lớn
NUM_WORKERS     = 2                 # Kaggle hỗ trợ multiprocessing
EPOCHS          = 50                # tăng lên 100 cho training đầy đủ
MIXED_PRECISION = True

LR              = 6e-5
WEIGHT_DECAY    = 0.01
BETAS           = (0.9, 0.999)
GRAD_CLIP       = 1.0

BCE_POS_WEIGHT  = 3.0               # weight cho class oil (class imbalance)
DICE_WEIGHT     = 0.5               # 0.5*Dice + 0.5*BCE
DICE_SMOOTH     = 1e-6

CHECKPOINT_DIR  = Path('/kaggle/working/checkpoints')
PREDICTION_DIR  = Path('/kaggle/working/predictions')
LOG_PATH        = Path('/kaggle/working/training_log.csv')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset paths ─────────────────────────────────────────────────────────────
TRAIN_IMG_DIR  = DATASET_ROOT / 'images' / 'images' / 'train'
TRAIN_MASK_DIR = DATASET_ROOT / 'masks'  / 'masks'  / 'train'
VAL_IMG_DIR    = DATASET_ROOT / 'images' / 'images' / 'val'
VAL_MASK_DIR   = DATASET_ROOT / 'masks'  / 'masks'  / 'val'

# Xác nhận paths tồn tại
for p, name in [(TRAIN_IMG_DIR,'train_img'), (TRAIN_MASK_DIR,'train_mask'),
                (VAL_IMG_DIR,'val_img'), (VAL_MASK_DIR,'val_mask')]:
    exists = p.exists()
    count  = len(list(p.glob('*.png'))) if exists else 0
    print(f'  {name:15s}: {str(p)[-50:]} | exists={exists} | n_png={count}')

## 3. EDA — Khám phá Dataset

In [ ]:
# ── Thống kê cơ bản ───────────────────────────────────────────────────────────
def compute_dataset_stats(image_paths: List[Path]) -> dict:
    """Tính thống kê ảnh SAR."""
    stats = {'heights': [], 'widths': [], 'means': []}
    for p in image_paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            stats['heights'].append(img.shape[0])
            stats['widths'].append(img.shape[1])
            stats['means'].append(img.mean())
    return stats

def compute_oil_ratio(mask_paths: List[Path]) -> List[float]:
    """Tính tỷ lệ pixel dầu trên mỗi mask."""
    ratios = []
    for p in mask_paths:
        msk = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if msk is not None:
            ratios.append((msk > 127).sum() / msk.size)
    return ratios

train_imgs  = sorted(TRAIN_IMG_DIR.glob('*.png'))
train_masks = sorted(TRAIN_MASK_DIR.glob('*.png'))
val_imgs    = sorted(VAL_IMG_DIR.glob('*.png'))
val_masks   = sorted(VAL_MASK_DIR.glob('*.png'))

print(f'Train: {len(train_imgs)} ảnh | Val: {len(val_imgs)} ảnh')

train_stats = compute_dataset_stats(train_imgs)
oil_ratios  = compute_oil_ratio(train_masks)

print(f'\nKích thước ảnh train:')
print(f'  H: [{min(train_stats["heights"])}, {max(train_stats["heights"])}]')
print(f'  W: [{min(train_stats["widths"])}, {max(train_stats["widths"])}]')
print(f'\nPixel intensity (train):')
print(f'  Mean: {np.mean(train_stats["means"]):.1f} | Std: {np.std(train_stats["means"]):.1f}')
print(f'\nOil ratio (train):')
print(f'  Mean: {np.mean(oil_ratios)*100:.2f}% | Max: {np.max(oil_ratios)*100:.2f}%')

In [ ]:
# ── Visualize mẫu ảnh ─────────────────────────────────────────────────────────
def show_samples(img_paths: List[Path], mask_paths: List[Path], n: int = 5):
    """Hiển thị n mẫu ảnh SAR cùng ground truth mask."""
    indices = random.sample(range(len(img_paths)), min(n, len(img_paths)))
    fig, axes = plt.subplots(2, len(indices), figsize=(4*len(indices), 8))
    
    for col, idx in enumerate(indices):
        img = cv2.imread(str(img_paths[idx]),  cv2.IMREAD_GRAYSCALE)
        msk = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        
        axes[0, col].imshow(img, cmap='gray')
        axes[0, col].set_title(f'SAR\n{img_paths[idx].stem[:15]}', fontsize=9)
        axes[0, col].axis('off')
        
        axes[1, col].imshow((msk > 127).astype(np.uint8), cmap='Reds', vmin=0, vmax=1)
        axes[1, col].set_title(f'Oil Mask\nratio={((msk>127).sum()/msk.size)*100:.1f}%', fontsize=9)
        axes[1, col].axis('off')
    
    fig.suptitle('Mẫu ảnh SAR & Ground Truth Mask', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/sample_images.png', dpi=120, bbox_inches='tight')
    plt.show()

show_samples(train_imgs, train_masks, n=5)

# Phân bố oil ratio
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(oil_ratios, bins=30, color='tomato', edgecolor='white', alpha=0.8)
axes[0].set_title('Phân bố tỷ lệ pixel dầu (training)')
axes[0].set_xlabel('Tỷ lệ oil pixel')
axes[0].set_ylabel('Số ảnh')
axes[0].grid(True, alpha=0.3)

# Số ảnh có oil vs không
has_oil = sum(1 for r in oil_ratios if r > 0)
axes[1].bar(['Có dầu', 'Không dầu'], [has_oil, len(oil_ratios)-has_oil],
            color=['tomato', 'steelblue'], edgecolor='white')
axes[1].set_title('Ảnh có/không có vùng dầu')
axes[1].set_ylabel('Số ảnh')
for i, v in enumerate([has_oil, len(oil_ratios)-has_oil]):
    axes[1].text(i, v+0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/distribution.png', dpi=120)
plt.show()

## 4. Tiền xử lý & Augmentation

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MASK_THRESHOLD = 127


def load_sar_image(image_path: Union[str, Path], img_size: Tuple[int,int]=(512,512)) -> np.ndarray:
    """Đọc ảnh SAR grayscale và resize. → (H, W) uint8"""
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f'Không đọc được: {image_path}')
    H, W = img_size
    if img.shape[:2] != (H, W):
        img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)
    return img


def load_mask(mask_path: Union[str, Path], img_size: Tuple[int,int]=(512,512)) -> np.ndarray:
    """Đọc mask, resize (nearest) và binarize. → (H, W) uint8 {0,1}"""
    msk = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if msk is None:
        raise IOError(f'Không đọc được mask: {mask_path}')
    H, W = img_size
    if msk.shape[:2] != (H, W):
        msk = cv2.resize(msk, (W, H), interpolation=cv2.INTER_NEAREST)
    return (msk > MASK_THRESHOLD).astype(np.uint8)


def to_rgb(gray: np.ndarray) -> np.ndarray:
    """Gray (H,W) → RGB (H,W,3) uint8"""
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


def normalize_imagenet(image_rgb: np.ndarray) -> np.ndarray:
    """Chuẩn hóa theo ImageNet stats. → (H,W,3) float32"""
    return (image_rgb.astype(np.float32)/255.0 - IMAGENET_MEAN) / IMAGENET_STD


def preprocess_for_inference(image_path: Union[str,Path], img_size=(512,512)) -> torch.Tensor:
    """Pipeline đầy đủ: path → tensor (1,3,H,W) float32"""
    img = load_sar_image(image_path, img_size)
    rgb = to_rgb(img)
    nrm = normalize_imagenet(rgb)
    t   = torch.from_numpy(nrm.transpose(2,0,1)).float()  # (3,H,W)
    return t.unsqueeze(0)  # (1,3,H,W)


def postprocess_mask(logits: torch.Tensor) -> np.ndarray:
    """logits (B,C,H,W) → mask (H,W) uint8"""
    return logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)


print('✓ Preprocessing functions defined')

In [ ]:
# ── Augmentation pipelines ────────────────────────────────────────────────────
def get_train_transforms(img_size=IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ElasticTransform(alpha=120, sigma=6, p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.2),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225), max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225), max_pixel_value=255.0),
        ToTensorV2(),
    ])


print('✓ Transforms defined')

In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class OilSpillDataset(Dataset):
    """Binary SAR oil spill segmentation dataset."""

    def __init__(self, img_dir: Path, mask_dir: Path,
                 transform=None, img_size=IMG_SIZE):
        self.img_paths  = sorted(Path(img_dir).glob('*.png'))
        self.mask_paths = sorted(Path(mask_dir).glob('*.png'))
        if not self.img_paths:
            raise FileNotFoundError(f'Không có ảnh trong: {img_dir}')
        if len(self.img_paths) != len(self.mask_paths):
            raise ValueError(f'Img/mask count mismatch: {len(self.img_paths)} vs {len(self.mask_paths)}')
        self.transform = transform
        self.img_size  = img_size

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img     = load_sar_image(self.img_paths[idx],  self.img_size)
        msk     = load_mask(self.mask_paths[idx], self.img_size)
        img_rgb = to_rgb(img)

        if self.transform is not None:
            aug = self.transform(image=img_rgb, mask=msk)
            return aug['image'], aug['mask'].long()
        else:
            nrm = normalize_imagenet(img_rgb)
            t   = torch.from_numpy(nrm.transpose(2,0,1)).float()
            m   = torch.from_numpy(msk).long()
            return t, m


# Tạo datasets và dataloaders
train_ds = OilSpillDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=get_train_transforms())
val_ds   = OilSpillDataset(VAL_IMG_DIR,   VAL_MASK_DIR,   transform=get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} ảnh → {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} ảnh → {len(val_loader)} batches')

# Kiểm tra batch shape
imgs, masks = next(iter(train_loader))
print(f'Batch image shape: {imgs.shape}   dtype: {imgs.dtype}')
print(f'Batch mask  shape: {masks.shape}  dtype: {masks.dtype}')
print(f'Mask unique values: {torch.unique(masks).tolist()}')

## 5. Model SegFormer

In [ ]:
# ── SegFormer Model ───────────────────────────────────────────────────────────
class OilSpillSegFormer(nn.Module):
    """
    SegFormer wrapper cho binary oil spill segmentation.
    HuggingFace SegFormer output H/4×W/4 → upsample bilinear về H×W.
    """

    def __init__(self, backbone=BACKBONE, num_classes=NUM_CLASSES, pretrained=PRETRAINED):
        super().__init__()
        if pretrained:
            self.model = SegformerForSemanticSegmentation.from_pretrained(
                backbone, num_labels=num_classes, ignore_mismatched_sizes=True
            )
        else:
            cfg = SegformerConfig.from_pretrained(backbone, num_labels=num_classes)
            self.model = SegformerForSemanticSegmentation(cfg)
        self.num_classes = num_classes

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:    pixel_values: (B, 3, H, W) normalized tensor
        Returns: logits:       (B, num_classes, H, W)
        """
        logits = self.model(pixel_values=pixel_values).logits  # (B, C, H/4, W/4)
        return F.interpolate(logits, size=pixel_values.shape[-2:], mode='bilinear', align_corners=False)

    @torch.no_grad()
    def predict_mask(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Inference → (B, H, W) int64 class map {0, 1}"""
        self.eval()
        return self.forward(pixel_values).argmax(dim=1)


# Build model
model = OilSpillSegFormer().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable/1e6:.1f}M')

# Sanity check forward pass
with torch.no_grad():
    test_out = model(imgs[:2].to(device))
print(f'Output shape: {test_out.shape}   (expected: [2, {NUM_CLASSES}, {IMG_SIZE[0]}, {IMG_SIZE[1]}])')

## 6. Loss Functions & Metrics

In [ ]:
# ── Dice Loss ─────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=DICE_SMOOTH, num_classes=NUM_CLASSES):
        super().__init__()
        self.smooth = smooth
        self.num_classes = num_classes

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        oh    = F.one_hot(targets, self.num_classes).permute(0,3,1,2).float()
        dims  = (0, 2, 3)
        inter = (probs * oh).sum(dims)
        card  = (probs + oh).sum(dims)
        dice  = (2*inter + self.smooth) / (card + self.smooth)
        return 1 - dice.mean()


# ── BCEDice Loss ──────────────────────────────────────────────────────────────
class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=BCE_POS_WEIGHT, dice_w=DICE_WEIGHT):
        super().__init__()
        self.dice_w    = dice_w
        self.dice_loss = DiceLoss()
        self.register_buffer('pos_weight', torch.tensor([pos_weight], dtype=torch.float32))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        dice = self.dice_loss(logits, targets)
        bce  = F.binary_cross_entropy_with_logits(
            logits[:,1], targets.float(), pos_weight=self.pos_weight
        )
        return self.dice_w * dice + (1 - self.dice_w) * bce


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(preds: np.ndarray, targets: np.ndarray,
                    smooth=1e-7) -> Dict[str, float]:
    preds, targets = preds.flatten(), targets.flatten()
    results = {}
    for cls, name in [(0,'bg'), (1,'oil')]:
        p, t   = (preds==cls), (targets==cls)
        tp     = (p & t).sum()
        fp, fn = (p & ~t).sum(), (~p & t).sum()
        results[f'iou_{name}']  = float((tp+smooth)/(tp+fp+fn+smooth))
        results[f'dice_{name}'] = float((2*tp+smooth)/(2*tp+fp+fn+smooth))

    p_oil, t_oil = (preds==1), (targets==1)
    tp = (p_oil & t_oil).sum()
    results['precision_oil'] = float((tp+smooth)/((p_oil.sum())+smooth))
    results['recall_oil']    = float((tp+smooth)/((t_oil.sum())+smooth))
    results['mean_iou']      = (results['iou_bg'] + results['iou_oil']) / 2
    results['mean_dice']     = (results['dice_bg'] + results['dice_oil']) / 2
    return results


criterion = BCEDiceLoss().to(device)
print('✓ Loss functions & metrics defined')

## 7. Training Loop

In [ ]:
# ── Optimizer & Scheduler ─────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=BETAS)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler    = GradScaler() if MIXED_PRECISION and device.type == 'cuda' else None

# ── CSV logging ───────────────────────────────────────────────────────────────
if not LOG_PATH.exists():
    with open(LOG_PATH, 'w', newline='') as f:
        csv.writer(f).writerow([
            'epoch','train_loss','val_loss',
            'mean_iou','mean_dice','iou_oil','dice_oil','precision_oil','recall_oil'
        ])


def train_epoch(model, loader, optimizer, criterion, scaler, device) -> float:
    """Training epoch → average loss."""
    model.train()
    total = 0.0
    for imgs, masks in tqdm(loader, desc='  Train', leave=False, ncols=80):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        if scaler:
            with autocast():
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(imgs), masks)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def val_epoch(model, loader, criterion, device) -> Tuple[float, Dict]:
    """Validation epoch → (avg_loss, metrics_dict)."""
    model.eval()
    total = 0.0
    all_preds, all_gts = [], []
    for imgs, masks in tqdm(loader, desc='  Val  ', leave=False, ncols=80):
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        total += criterion(logits, masks).item()
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_gts.append(masks.cpu().numpy())
    metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_gts))
    return total / len(loader), metrics


print('✓ Training loop defined')
print(f'  Optimizer: AdamW | LR={LR:.0e} | WD={WEIGHT_DECAY}')
print(f'  Scheduler: CosineAnnealingLR (T_max={EPOCHS})')
print(f'  AMP: {scaler is not None}')

In [ ]:
# ── Main Training Loop ────────────────────────────────────────────────────────
best_dice    = 0.0
history      = {'train_loss': [], 'val_loss': [], 'mean_dice': [], 'iou_oil': []}

for epoch in range(1, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n{'='*65}")
    print(f"Epoch {epoch:3d}/{EPOCHS}  |  LR: {current_lr:.2e}")
    print('='*65)

    train_loss             = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_metrics  = val_epoch(model, val_loader, criterion, device)

    scheduler.step()

    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['mean_dice'].append(val_metrics['mean_dice'])
    history['iou_oil'].append(val_metrics['iou_oil'])

    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Val   loss: {val_loss:.4f} | "
          f"mIoU: {val_metrics['mean_iou']:.4f} | "
          f"Dice: {val_metrics['mean_dice']:.4f} | "
          f"Oil IoU: {val_metrics['iou_oil']:.4f}")

    # CSV
    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch, f'{train_loss:.6f}', f'{val_loss:.6f}',
            val_metrics.get('mean_iou',''), val_metrics.get('mean_dice',''),
            val_metrics.get('iou_oil',''),  val_metrics.get('dice_oil',''),
            val_metrics.get('precision_oil',''), val_metrics.get('recall_oil','')
        ])

    # Save best checkpoint
    if val_metrics['mean_dice'] > best_dice:
        best_dice = val_metrics['mean_dice']
        torch.save({
            'epoch': epoch, 'val_dice': best_dice,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, CHECKPOINT_DIR / 'best_model.pth')
        print(f'  ★ Best checkpoint saved (Dice={best_dice:.4f})')

print(f'\n✓ Training complete! Best Dice: {best_dice:.4f}')

## 8. Đánh giá & Visualization

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Epoch')

axes[1].plot(history['mean_dice'], label='Mean Dice', color='mediumseagreen')
axes[1].set_title('Dice Coefficient'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0, 1)

axes[2].plot(history['iou_oil'], label='Oil IoU', color='darkorange')
axes[2].set_title('Oil Spill IoU'); axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Epoch'); axes[2].set_ylim(0, 1)

fig.suptitle(f'SegFormer ({BACKBONE}) — Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print(f'Best Dice: {max(history["mean_dice"]):.4f} | Best Oil IoU: {max(history["iou_oil"]):.4f}')

In [ ]:
# ── Load best checkpoint & evaluate ─────────────────────────────────────────
ckpt = torch.load(CHECKPOINT_DIR / 'best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint: epoch={ckpt['epoch']}, dice={ckpt['val_dice']:.4f}")

val_loss_final, final_metrics = val_epoch(model, val_loader, criterion, device)

print('\n' + '='*50)
print('  KẾT QUẢ ĐÁNH GIÁ (Best Checkpoint)')
print('='*50)
for k, v in final_metrics.items():
    print(f'  {k:20s}: {v:.4f}')
print('='*50)

In [ ]:
# ── Visualization kết quả ─────────────────────────────────────────────────────
def overlay_prediction(sar_gray, pred_mask, alpha=0.45, oil_color=(255,80,80)):
    """Overlay mask dự đoán lên ảnh SAR."""
    rgb = cv2.cvtColor(sar_gray, cv2.COLOR_GRAY2RGB)
    ovr = rgb.copy()
    ovr[pred_mask == 1] = oil_color
    return cv2.addWeighted(rgb, 1-alpha, ovr, alpha, 0)


model.eval()
sample_imgs  = sorted(VAL_IMG_DIR.glob('*.png'))[:8]
sample_masks = sorted(VAL_MASK_DIR.glob('*.png'))[:8]

fig, axes = plt.subplots(3, len(sample_imgs), figsize=(3*len(sample_imgs), 9))
row_labels = ['SAR Image', 'Ground Truth', 'Prediction']

for col, (ip, mp) in enumerate(zip(sample_imgs, sample_masks)):
    sar  = load_sar_image(ip, IMG_SIZE)
    gt   = load_mask(mp, IMG_SIZE)

    tensor = preprocess_for_inference(ip, IMG_SIZE).to(device)
    pred   = postprocess_mask(model(tensor))

    oil_dice = float((2*((pred==1)&(gt==1)).sum() + 1e-7) /
                     (2*((pred==1)&(gt==1)).sum() + ((pred==1)&(gt==0)).sum() +
                      ((pred==0)&(gt==1)).sum() + 1e-7))

    axes[0, col].imshow(sar, cmap='gray')
    axes[0, col].set_title(f'{ip.stem[:10]}', fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(gt, cmap='Reds', vmin=0, vmax=1)
    axes[1, col].axis('off')

    axes[2, col].imshow(overlay_prediction(sar, pred))
    axes[2, col].set_title(f'Dice={oil_dice:.2f}', fontsize=7)
    axes[2, col].axis('off')

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=90, labelpad=5)

red_patch = mpatches.Patch(color=(1,0.31,0.31), label='Oil spill')
fig.legend(handles=[red_patch], loc='lower right', fontsize=9)
fig.suptitle('SAR | Ground Truth | Predicted (with overlay)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/predictions_grid.png', dpi=150)
plt.show()

## 9. Inference & Export

In [ ]:
# ── Public inference API (chuẩn hóa) ─────────────────────────────────────────
def predict(image_path: Union[str, Path],
            img_size: Tuple[int,int] = IMG_SIZE) -> np.ndarray:
    """
    Dự đoán binary mask từ ảnh SAR.

    Args:
        image_path: Đường dẫn tới ảnh SAR PNG.
        img_size:   (H, W) resize target, mặc định IMG_SIZE.

    Returns:
        np.ndarray (H, W) uint8: {0=background, 1=oil spill}

    Example:
        >>> mask = predict('sar_image.png', img_size=(512, 512))
    """
    model.eval()
    tensor = preprocess_for_inference(image_path, img_size).to(device)
    with torch.no_grad():
        logits = model(tensor)
    return postprocess_mask(logits)


def predict_batch(image_dir: Union[str, Path],
                  img_size: Tuple[int,int] = IMG_SIZE,
                  batch_size: int = BATCH_SIZE) -> List[np.ndarray]:
    """
    Dự đoán mask cho toàn bộ ảnh trong thư mục.

    Args:
        image_dir:  Thư mục chứa ảnh SAR PNG.
        img_size:   (H, W) resize target.
        batch_size: Số ảnh mỗi batch.

    Returns:
        List[np.ndarray]: Danh sách mask (H, W) uint8.
    """
    model.eval()
    paths   = sorted(Path(image_dir).glob('*.png'))
    masks   = []
    for i in tqdm(range(0, len(paths), batch_size), desc='Inference'):
        batch  = paths[i:i+batch_size]
        tensors = torch.stack([
            preprocess_for_inference(p, img_size).squeeze(0) for p in batch
        ]).to(device)
        with torch.no_grad():
            preds = model(tensors).argmax(1)  # (B, H, W)
        masks.extend([preds[j].cpu().numpy().astype(np.uint8) for j in range(len(batch))])
    return masks


def predict_with_overlay(image_path: Union[str, Path],
                         img_size: Tuple[int,int] = IMG_SIZE,
                         alpha: float = 0.45) -> np.ndarray:
    """
    Dự đoán và trả về ảnh có overlay mask dầu loang.

    Args:
        image_path: Đường dẫn ảnh SAR.
        img_size:   (H, W) resize target.
        alpha:      Độ trong suốt overlay [0,1].

    Returns:
        np.ndarray (H, W, 3) uint8 RGB.

    Example:
        >>> img = predict_with_overlay('sar.png')
        >>> plt.imshow(img); plt.show()
    """
    mask = predict(image_path, img_size)
    sar  = load_sar_image(image_path, img_size)
    return overlay_prediction(sar, mask, alpha=alpha)


# Demo
demo_path = sample_imgs[0]
demo_mask = predict(demo_path, img_size=IMG_SIZE)
print(f'predict() output: shape={demo_mask.shape}, dtype={demo_mask.dtype}, unique={np.unique(demo_mask)}')

demo_overlay = predict_with_overlay(demo_path)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(cv2.imread(str(demo_path), cv2.IMREAD_GRAYSCALE), cmap='gray')
ax[0].set_title('SAR Input'); ax[0].axis('off')
ax[1].imshow(demo_overlay)
ax[1].set_title('predict_with_overlay()'); ax[1].axis('off')
plt.suptitle('API Demo', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Export predictions trên val set ───────────────────────────────────────────
PREDICTION_DIR.mkdir(exist_ok=True)

val_masks_pred = predict_batch(VAL_IMG_DIR, img_size=IMG_SIZE)
val_img_paths  = sorted(VAL_IMG_DIR.glob('*.png'))

for img_path, mask in zip(val_img_paths, val_masks_pred):
    sar  = load_sar_image(img_path, IMG_SIZE)
    ovr  = overlay_prediction(sar, mask)
    cv2.imwrite(
        str(PREDICTION_DIR / f'{img_path.stem}_overlay.png'),
        cv2.cvtColor(ovr, cv2.COLOR_RGB2BGR)
    )
    cv2.imwrite(
        str(PREDICTION_DIR / f'{img_path.stem}_mask.png'),
        (mask * 255).astype(np.uint8)
    )

print(f'✓ Đã lưu {len(val_img_paths)} predictions tại: {PREDICTION_DIR}')
print(f'  Checkpoint: {CHECKPOINT_DIR}/best_model.pth')
print(f'  Log:        {LOG_PATH}')

In [ ]:
# ── Tóm tắt kết quả ───────────────────────────────────────────────────────────
print('='*60)
print('  TÓM TẮT KẾT QUẢ TRAINING')
print('='*60)
print(f'  Backbone     : {BACKBONE}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Image size   : {IMG_SIZE}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Best Val Dice: {best_dice:.4f}')
print()
for k, v in final_metrics.items():
    print(f'  {k:20s}: {v:.4f}')
print('='*60)
print('\n  Output files:')
print(f'    /kaggle/working/best_model.pth        ← best checkpoint')
print(f'    /kaggle/working/training_log.csv      ← epoch metrics')
print(f'    /kaggle/working/training_curves.png   ← learning curves')
print(f'    /kaggle/working/predictions_grid.png  ← visual results')
print(f'    /kaggle/working/predictions/          ← {len(val_img_paths)} mask images')